# Call Center Confidence Model (MVP with Synthetic Data + BERT)

**Goal**: Predict confidence level of AI in ongoing conversation  
(high / medium / low) → used to decide escalation to human agent

**Model**: bert-base-uncased (stronger than DistilBERT)

**Approach**:
1. Generate synthetic dataset
2. Fine-tune BERT for 3-class classification
3. Evaluate & inference

Date: Jan 2026

In [1]:
!pip install -q transformers datasets torch scikit-learn pandas tqdm

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import Dataset as HFDataset

## 1. Create synthetic dataset

In [3]:
np.random.seed(42)
data = []

def add_example(customer_text, agent_text, confidence_label, score):
    text = f"Customer: {customer_text}\nAgent: {agent_text}"
    data.append({"text": text, "label": confidence_label, "score": score})

# HIGH confidence examples

add_example(
        "My package still hasn't arrived.",
        "I see it's delayed due to weather. I've expedited it and added a $10 credit. Expected delivery is now tomorrow.",
        "high", 0.91
    )

add_example(
        "The bill amount looks wrong.",
        "You're right, there was a billing error. I've corrected it and applied a $18.50 refund.",
        "high", 0.89
    )

add_example(
        "Internet keeps dropping.",
        "I've run diagnostics — the issue was with the upstream node. Connection is stable now.",
        "high", 0.94
    )

# MEDIUM confidence

add_example(
        "App crashes when I try to pay.",
        "Thanks for reporting. Which device and OS version? I'll check known issues.",
        "medium", 0.58
    )

add_example(
        "I want to change my plan.",
        "Sure, let me check options... What features are most important to you?",
        "medium", 0.64
    )

add_example(
        "My SIM isn't working abroad.",
        "Roaming looks enabled. Could you restart the phone and test again?",
        "medium", 0.52
    )

# LOW confidence (unhelpful/off-topic/confused agent)

add_example(
        "It's working now actually.",
        "Oh... um... so everything is fine? Anything is possible I guess...",
        "low", 0.19
    )

add_example(
        "All good now, thanks.",
        "Great! Wait... what was the issue again? Can you explain one more time?",
        "low", 0.14
    )

add_example(
        "Problem solved, thank you.",
        "Perfect! So would you like to hear about our new insurance add-on today?",
        "low", 0.27
    )

# Mixed: Mild frustration → good resolution

add_example(
        "This is really frustrating... been waiting 20 minutes.",
        "I sincerely apologize. I've processed everything manually — issue fully resolved.",
        "high", 0.85
    )

# Original patterns (you can keep or remove)

add_example(
        "My internet is not working since yesterday.",
        "I understand. I've restarted the line — it's back online now.",
        "high", 0.92
    )

add_example(
        "This app keeps crashing on Android 14.",
        "Hmm... Did you try clearing cache? Can you tell me the model?",
        "medium", 0.55
    )

add_example(
        "This is the THIRD time I'm calling! Nothing works!",
        "I'm really sorry... let me see what I can do...",
        "low", 0.18
    )

# Casual but helpful & successful
add_example(
    "Thanks a lot! It's working now.",
    "No worries bro, glad it's sorted! 😎",
    "high", 0.89
)
add_example(
    "It's fixed, thank you!",
    "Sweet! Happy to help dude.",
    "high", 0.87
)
add_example(
    "Wow thanks, problem solved.",
    "You're welcome mate, enjoy!",
    "high", 0.85
)
add_example(
    "Thanks so much!",
    "Anytime! Take care bro.",
    "high", 0.88
)
add_example(
    "Everything is working perfectly now, thanks!",
    "Legend! Cheers for letting me know 👊",
    "high", 0.90
)
add_example(
    "App is running smooth again, appreciate it.",
    "Good stuff! Enjoy the rest of your day fam.",
    "high", 0.86
)

# Very brief but positive & successful
add_example(
    "All good now.",
    "Awesome, cheers!",
    "high", 0.84
)
add_example(
    "It's working, thanks.",
    "Cool beans!",
    "high", 0.82
)
add_example(
    "Fixed, thank you.",
    "Sweet as!",
    "high", 0.83
)
add_example(
    "Problem gone. Thanks!",
    "No prob at all!",
    "high", 0.85
)
add_example(
    "Sorted now.",
    "Nice one!",
    "high", 0.81
)

df = pd.DataFrame(data)
print(f"Created synthetic dataset with {len(df)} examples")
print(df['label'].value_counts())
df.sample(8)

Created synthetic dataset with 24 examples
label
high      16
medium     4
low        4
Name: count, dtype: int64


,text,label,score
8,"Customer: Problem solved, thank you.\nAgent: P...",low,0.27
16,Customer: Thanks so much!\nAgent: Anytime! Tak...,high,0.88
0,Customer: My package still hasn't arrived.\nAg...,high,0.91
18,"Customer: App is running smooth again, appreci...",high,0.86
11,Customer: This app keeps crashing on Android 1...,medium,0.55
9,Customer: This is really frustrating... been w...,high,0.85
13,Customer: Thanks a lot! It's working now.\nAge...,high,0.89
1,Customer: The bill amount looks wrong.\nAgent:...,high,0.89


## 2. Prepare data for training

In [4]:
label2id = {"low": 0, "medium": 1, "high": 2}
id2label = {v: k for k, v in label2id.items()}

df['label_id'] = df['label'].map(label2id)

# train_df, test_df = train_test_split(
#     df, test_size=0.25, stratify=df['label_id'], random_state=12
# )
# Option A: Shuffle much more aggressively
train_df, test_df = train_test_split(
    df, 
    test_size=0.25, 
    stratify=df['label_id'], 
    random_state=np.random.randint(0, 10000)  # change seed every run
)

print(f"Train: {len(train_df)}   Test: {len(test_df)}")

Train: 18   Test: 6


In [5]:
# 1. Check for exact duplicates
train_texts = set(train_df['text'])
test_texts = set(test_df['text'])

exact_overlap = train_texts & test_texts
print(f"Exact duplicates between train & test: {len(exact_overlap)}")
if exact_overlap:
    print("\nSome examples that leaked:")
    for txt in list(exact_overlap)[:5]:
        print(txt[:100] + "...")

# 2. Check very high similarity (more realistic leakage)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer().fit(df['text'])
tfidf_all = vectorizer.transform(df['text'])

train_idx = train_df.index
test_idx = test_df.index

sim_matrix = cosine_similarity(
    tfidf_all[test_idx],
    tfidf_all[train_idx]
)

max_sim = sim_matrix.max(axis=1)
print("\nMax similarity of each test example to closest train example:")
print(pd.Series(max_sim).describe())

# How many test examples are extremely similar to train
print(f"Test examples with ≥95% similarity to some train: {(max_sim >= 0.95).sum()} / {len(max_sim)}")
print(f"Test examples with ≥90% similarity: {(max_sim >= 0.90).sum()} / {len(max_sim)}")

Exact duplicates between train & test: 0

Max similarity of each test example to closest train example:
count    6.000000
mean     0.261711
std      0.153112
min      0.137929
25%      0.163299
50%      0.214194
75%      0.286016
max      0.547659
dtype: float64
Test examples with ≥95% similarity to some train: 0 / 6
Test examples with ≥90% similarity: 0 / 6


In [6]:
# ===============================================
# Using bert-base-uncased (stronger model)
# ===============================================
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256                  # BERT handles longer context better
    )

train_dataset = HFDataset.from_pandas(train_df).map(tokenize_function, batched=True)
test_dataset  = HFDataset.from_pandas(test_df).map(tokenize_function, batched=True)

train_dataset = train_dataset.rename_column("label_id", "labels")
test_dataset  = test_dataset.rename_column("label_id", "labels")

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

## 3. Fine-tune BERT

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./confidence_bert_results",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
)

# training_args = TrainingArguments(
#     output_dir="./confidence_bert_results",
#     num_train_epochs=4,
#     per_device_train_batch_size=8,
#     gradient_accumulation_steps=2,           # effective batch size = 16 
#     per_device_eval_batch_size=16,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     logging_steps=30,
#     learning_rate=2e-5,
#     weight_decay=0.01,
#     warmup_ratio=0.1,
#     load_best_model_at_end=True,
#     metric_for_best_model="accuracy",
#     greater_is_better=True,
#     fp16=torch.cuda.is_available(),
#     report_to="none"
# )

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = (predictions == labels).mean()
    return {"accuracy": acc}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose "Don't visualize my results"


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.932164,0.666667
2,No log,0.863542,0.666667
3,No log,0.824092,0.666667
4,No log,0.827771,0.666667


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=12, training_loss=0.838512102762858, metrics={'train_runtime': 307.2364, 'train_samples_per_second': 0.234, 'train_steps_per_second': 0.039, 'total_flos': 9472083038208.0, 'train_loss': 0.838512102762858, 'epoch': 4.0})

## 4. Evaluate

In [8]:
eval_results = trainer.evaluate()
print(eval_results)

predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

print("\nClassification Report:")
print(classification_report(true_labels, pred_labels, target_names=["low", "medium", "high"]))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.8277713656425476, 'eval_accuracy': 0.6666666666666666, 'eval_runtime': 5.9413, 'eval_samples_per_second': 1.01, 'eval_steps_per_second': 0.168, 'epoch': 4.0}

Classification Report:
              precision    recall  f1-score   support

         low       0.00      0.00      0.00         1
      medium       0.00      0.00      0.00         1
        high       0.67      1.00      0.80         4

    accuracy                           0.67         6
   macro avg       0.22      0.33      0.27         6
weighted avg       0.44      0.67      0.53         6



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## 5. Inference examples

In [9]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

examples = [
    "Customer: You are wasting my time! Fix this NOW!\nAgent: Ma'am, please calm down...",
    "Customer: Thanks for the quick help!\nAgent: My pleasure! Anything else I can assist with?",
    "Customer: I don't know... maybe it's the cable?\nAgent: Possible. Can you try another port?",
    "Customer: It's working now actually.\nAgent: Well then, anything is possible I guess...",
    "Customer: Problem solved.\nAgent: Great! Would you like our new insurance add-on?",
    "Customer: Are you sure?\nAgent: Yeah, It is the correct way",
    "Customer: Thanks a lot, It's working\nAgent: Welcome sir"
]

for ex in examples:
    result = classifier(ex, truncation=True, max_length=256)[0]
    print(f"\nText:\n{ex}\n→ {result['label']} (score: {result['score']:.3f})")

Device set to use cpu



Text:
Customer: You are wasting my time! Fix this NOW!
Agent: Ma'am, please calm down...
→ high (score: 0.593)

Text:
Customer: Thanks for the quick help!
Agent: My pleasure! Anything else I can assist with?
→ high (score: 0.541)

Text:
Customer: I don't know... maybe it's the cable?
Agent: Possible. Can you try another port?
→ high (score: 0.613)

Text:
Customer: It's working now actually.
Agent: Well then, anything is possible I guess...
→ high (score: 0.626)

Text:
Customer: Problem solved.
Agent: Great! Would you like our new insurance add-on?
→ high (score: 0.577)

Text:
Customer: Are you sure?
Agent: Yeah, It is the correct way
→ high (score: 0.674)

Text:
Customer: Thanks a lot, It's working
Agent: Welcome sir
→ high (score: 0.670)
